# Lesson 6.3 - Model Deployment & Serving (FastAPI toy service)

This notebook demonstrates a lightweight deployment flow: train a model, persist artifact, define a FastAPI inference endpoint, and test request/response behavior.

## Objectives

- Package a trained model for serving.
- Expose a prediction API using FastAPI.
- Validate endpoint contract with local tests.
- Map the toy setup to production serving architecture.

## Setup

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

SEED = 42
ARTIFACT_DIR = Path("artifacts/lesson6_3")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## Section A - Train and Persist a Model Artifact

In [2]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

model = RandomForestClassifier(n_estimators=200, random_state=SEED)
model.fit(X_train, y_train)

acc = accuracy_score(y_test, model.predict(X_test))

model_path = ARTIFACT_DIR / "iris_rf.joblib"
meta_path = ARTIFACT_DIR / "model_metadata.json"
joblib.dump(model, model_path)

meta = {
    "model_type": "RandomForestClassifier",
    "feature_names": list(X.columns),
    "class_names": iris.target_names.tolist(),
    "accuracy_test": float(acc),
}
meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")

meta

{'model_type': 'RandomForestClassifier',
 'feature_names': ['sepal length (cm)',
  'sepal width (cm)',
  'petal length (cm)',
  'petal width (cm)'],
 'class_names': ['setosa', 'versicolor', 'virginica'],
 'accuracy_test': 0.9}

## Section B - Define FastAPI Inference Service

In [ ]:
from textwrap import dedent

app_code = dedent(
    '''    from __future__ import annotations

    import joblib
    from fastapi import FastAPI
    from pydantic import BaseModel, Field

    from pathlib import Path

    MODEL_PATH = Path(__file__).with_name("iris_rf.joblib")
    model = joblib.load(MODEL_PATH)

    app = FastAPI(title="Iris Model Service", version="1.0.0")


    class IrisRequest(BaseModel):
        sepal_length_cm: float = Field(..., ge=0)
        sepal_width_cm: float = Field(..., ge=0)
        petal_length_cm: float = Field(..., ge=0)
        petal_width_cm: float = Field(..., ge=0)


    @app.get("/health")
    def health() -> dict:
        return {{"status": "ok"}}


    @app.post("/predict")
    def predict(payload: IrisRequest) -> dict:
        x = [[
            payload.sepal_length_cm,
            payload.sepal_width_cm,
            payload.petal_length_cm,
            payload.petal_width_cm,
        ]]
        pred = int(model.predict(x)[0])
        probs = model.predict_proba(x)[0].tolist()
        return {{"prediction": pred, "probabilities": probs}}
    '''
).strip()

app_path = ARTIFACT_DIR / "app.py"
app_path.write_text(app_code, encoding="utf-8")

print(app_code[:700] + "\n...\n")
print(f"app saved to: {app_path}")


## Section C - Endpoint Test (if FastAPI installed)

In [4]:
test_result = {}

try:
    import importlib.util
    from fastapi.testclient import TestClient

    spec = importlib.util.spec_from_file_location("lesson63_app", app_path)
    mod = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(mod)

    client = TestClient(mod.app)

    health = client.get("/health")
    pred = client.post(
        "/predict",
        json={
            "sepal_length_cm": 5.1,
            "sepal_width_cm": 3.5,
            "petal_length_cm": 1.4,
            "petal_width_cm": 0.2,
        },
    )

    test_result = {
        "health_status_code": health.status_code,
        "health_response": health.json(),
        "predict_status_code": pred.status_code,
        "predict_response": pred.json(),
    }
except Exception as exc:
    test_result = {
        "status": "FastAPI test skipped",
        "reason": str(exc),
        "hint": "Install fastapi and uvicorn to run full local service tests.",
    }

test_result

{'status': 'FastAPI test skipped',
 'reason': "No module named 'fastapi'",
 'hint': 'Install fastapi and uvicorn to run full local service tests.'}

## Section D - How to Run the Service Locally

If dependencies are available, run:

```bash
uvicorn artifacts.lesson6_3.app:app --reload --port 8000
```

Then call:

```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H "content-type: application/json" \
  -d '{"sepal_length_cm":5.1,"sepal_width_cm":3.5,"petal_length_cm":1.4,"petal_width_cm":0.2}'
```

## Connect to Theory

- Model artifact + metadata represent packaging and versioning basics.
- FastAPI endpoint represents online real-time serving.
- TestClient validation mirrors CI endpoint contract tests.
- In production, this service is containerized, deployed with staged traffic rollout, and monitored via SLOs.

## Business Case Studies & Exceptions

### Case 1: Recommendation API rollout

- **Pattern**: deploy model service with canary traffic and KPI + latency gates.
- **Benefit**: safer releases and faster rollback.

### Case 2: Compliance-heavy credit decision service

- **Pattern**: model registry approvals + immutable artifacts + audited promotion.
- **Benefit**: traceability for regulatory review.

### Exceptions

- Not all models need APIs; many business workflows are better served by batch scoring.
- Aggressive autoscaling can increase cost if request batching and caching are ignored.

## Interview Questions & Answers

1. **Q: What does model deployment include beyond the model file?**
   **A:** Runtime dependencies, inference contract, observability, scaling, and rollback strategy.

2. **Q: Online vs batch serving?**
   **A:** Online serves request-time predictions; batch computes periodic predictions for datasets.

3. **Q: Why use FastAPI for model serving?**
   **A:** It provides typed request validation, OpenAPI docs, and straightforward Python integration.

4. **Q: What is a health endpoint for?**
   **A:** Infrastructure and orchestration systems use it to verify service readiness/liveness.

5. **Q: Why test endpoint contracts in CI?**
   **A:** To catch schema and behavior regressions before deployment.

6. **Q: What is blue-green deployment?**
   **A:** Switching traffic between two environments to minimize risk and speed rollback.

7. **Q: What is canary deployment?**
   **A:** Gradual traffic shift to a new version while monitoring quality and reliability metrics.

8. **Q: How do you version model services?**
   **A:** Version both model artifacts and service/container releases, linked by lineage metadata.

9. **Q: What is the most common serving reliability mistake?**
   **A:** Ignoring tail latency and focusing only on average response time.

10. **Q: How do you secure prediction endpoints?**
   **A:** Enforce auth, authorization, input validation, rate limits, and audit logging.

11. **Q: What should trigger rollback?**
   **A:** SLO breaches (latency, error rate), business KPI drops, or anomaly detection alerts.

12. **Q: How do you evolve this toy API to production?**
   **A:** Containerize, add auth/logging/metrics, deploy with staged rollout, and integrate registry-driven promotion.
